# GAVN: square-token geometric action-value model

One notebook = one GPU. Duplicate this notebook to use two concurrent kernels on one Kaggle account. Change `RUN_ID`, `DIM`, and `SEED` so each job has an independent HF prefix. The 192-wide configuration is the ~3M candidate; 224-wide is the ~5M candidate.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time
REPO = Path('/kaggle/working/chess-slm-benchmark')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Vedang-P/chess-slm-benchmark.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', 'origin/main'], check=True)
SL_REPO = Path('/kaggle/working/searchless_chess')
if not SL_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/google-deepmind/searchless_chess.git', str(SL_REPO)], check=True)
HF_SHARDS = 'chessbench-full-build'  # 8 shards on HF, 5GB peak, no 25GB assemble
assert (REPO / 'scripts/train_gavn.py').exists(), 'clone failed'
# HF token with retry (Kaggle Secrets service flaps)
for _ in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_WRITE_TOKEN'] = UserSecretsClient().get_secret('HF_WRITE_TOKEN')
        break
    except Exception as exc:
        print(f'HF secret retry in 5s: {exc}')
        time.sleep(5)
if not os.environ.get('HF_WRITE_TOKEN'):
    print('WARNING: HF_WRITE_TOKEN not available after retries, checkpoints will be local only')
import subprocess as _sp
_sp.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'python-chess'], check=True)
os.chdir(REPO)


In [ ]:
# Mandatory smoke test before spending GPU-hours (sharded fetch).
smoke = Path('/kaggle/working/gavn-smoke')
cmd = [sys.executable, 'scripts/train_gavn.py', '--hf-shards', HF_SHARDS, '--outdir', str(smoke), '--sl-repo', str(SL_REPO), '--dim', '96', '--layers', '2', '--heads', '4', '--batch', '32', '--steps', '20', '--max-records', '4096', '--ckpt-every', '20']
subprocess.run(cmd, check=True)
print('GAVN smoke test passed')


In [ ]:
# Production configuration. Recommended first pair: 3M and 5M, same seed.
RUN_ID = 'account1-gavn-3m-seed0'
DIM = 192
LAYERS = 8
HEADS = 8
SEED = 0
STEPS = 30000
BIAS_MODE = 'both'  # both | fixed | dynamic | none (geometry ablation)
RESUME = True
OUT = Path('/kaggle/working') / RUN_ID
cmd = [sys.executable, 'scripts/train_gavn.py', '--hf-shards', HF_SHARDS, '--outdir', str(OUT), '--sl-repo', str(SL_REPO), '--dim', str(DIM), '--layers', str(LAYERS), '--heads', str(HEADS), '--batch', '1024', '--steps', str(STEPS), '--lr', '0.0003', '--warmup', '1000', '--temperature', '1.0', '--bias-mode', BIAS_MODE, '--w-dist', '1.0', '--w-q', '0.5', '--w-ce', '0.25', '--seed', str(SEED), '--ckpt-every', '2000', '--hf-repo', 'vedangfake/chess-slm-benchmark', '--hf-run', RUN_ID, '--hf-upload-every', '1800']
if RESUME:
    cmd.append('--resume-from-hf')
print(' '.join(cmd))
subprocess.run(cmd, check=True)


The first production run is deliberately a distillation/geometry measurement, not the final paper result. It must be evaluated on held-out training positions before touching the frozen MATE and puzzle tests.